### MVP for LinkBikeNet

In [ ]:
# imports
import os
import osmnx as ox
import networkx as nx
import pandas as pd

from linkbikenet.functions import *

In [ ]:
# config
city_name = 'Budapest'
ox.settings.useful_tags_way = ["highway", "cycleway", "cycleway:right", "cycleway:left", "cycleway:both", "cyclestreet"]

In [ ]:
# fetch street network from OSM
g = ox.graph_from_place(
        city_name, network_type='all_public', simplify=False, retain_all=True
    )
g = ox.simplify_graph(
    g,
    edge_attrs_differ=['cycleway', 'highway', 'cycleway:right', 'cycleway:left', 'cycleway:both'],
)
ox.plot_graph(g);

In [ ]:
g = ox.project_graph(g, to_crs="3857")

### Tag edges that have protected bicycle infrastructure

In [ ]:
g = map_edges_to_bike_infrastructure(g)

### Drop parallel edges and convert to undirected graph

In [ ]:
# finding parallel edges and dropping them
edges_to_drop = find_edges_to_drop(g)
g.remove_edges_from(edges_to_drop)

In [ ]:
# Capital-G: the Graph() object we will be working with from now on
G = nx.Graph(g)

### Build graph with all protected infrastructure

In [ ]:
edges = [
    (u, v)
    for u, v, data in G.edges(data=True)
    if data.get("pbi") == 1
]

H = G.edge_subgraph(edges).copy()

In [ ]:
wcc = [H.subgraph(c).copy() for c in sorted(nx.connected_components(H), key=lambda c: sum([l[-1] for l in H.subgraph(c).copy().edges.data('length')]), reverse=True)]

In [ ]:
to_iterate = len(wcc) -1
closest_pairs = []
for i in range(to_iterate):
    wcc = [H.subgraph(c).copy() for c in sorted(nx.connected_components(H), key=lambda c: sum([l[-1] for l in H.subgraph(c).copy().edges.data('length')]), reverse=True)]
    pair = pair_between_largest_components(wcc)
    closest_pairs.append(pair)
    H.add_edge(pair[0], pair[1], length=0)

In [ ]:
# find paths between nodes
paths = []
for pair in closest_pairs:
    try:
        path = nx.shortest_path(G, pair[0], pair[1], weight='length')
    except nx.NetworkXNoPath:
        continue
    paths.append(path)

In [ ]:
edges_gdf = graph_edges_to_gdf(G)

In [ ]:
df = pd.DataFrame()
df['nodelist'] = paths

In [ ]:
df['edge_list'] = df.nodelist.apply(lambda x: get_correct_edgetuples(edges_gdf, x))

In [ ]:
gdf = create_gdf_with_geoms(df, edges_gdf)

In [ ]:
len(gdf)

In [ ]:
# reset Graph
H = G.edge_subgraph(edges).copy()

# calculating connectivity metrics
print("Calculating connectivity metrics...")
network_lengths = []
lcc_lengths = []
edge_lengths = gdf['geometry'].length

for i in range(len(gdf)):
    H.add_edge(closest_pairs[i][0], closest_pairs[i][1], length=edge_lengths[i])
    total, largest = calculate_network_statistics(H)
    network_lengths.append(total)
    lcc_lengths.append(largest)

In [ ]:
gdf['network_length'] = network_lengths
gdf['lcc_length'] = lcc_lengths

In [ ]:
os.makedirs("./results/", exist_ok=True)
gdf.geometry = gdf.geometry.set_precision(grid_size=1)
gdf.to_file("./results/budapest.geojson", driver="GeoJSON")

In [ ]:
edges_pbi_gdf = edges_gdf[edges_gdf["pbi"] == 1]

In [ ]:
edges_pbi_gdf.geometry = edges_pbi_gdf.geometry.set_precision(grid_size=1)
edges_pbi_gdf.to_file("./results/Budapest_bike_network.geojson", driver="GeoJSON")